# 05. Construção da Base de Avaliação
Organizamos perguntas e referências por norma e dispositivo, sem depender das estratégias de chunking. O catálogo inicial é um rascunho para revisão, não um conjunto de referência juridicamente validado.

Perguntas sem resposta exigem revisão do corpus completo: estar fora da CLT não significa estar fora da base, que também contém Código Civil, CPC e outras normas. Os splits iniciais são preservados e verificamos sobreposição de perguntas e grupos.

In [22]:
from pathlib import Path
import pandas as pd
import json
import re
import hashlib
import unicodedata
from datetime import datetime, timezone

def find_project_root(start: Path) -> Path:
    for base in (start, *start.parents):
        if (base / 'data/processed/legislacao/legislacao_artigos_completos.json').is_file():
            return base
    raise FileNotFoundError('Base de artigos não encontrada. Execute as etapas complementares do notebook 03.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROCESSED_DIR = PROJECT_ROOT / 'data/processed/legislacao'
EVAL_DIR = PROJECT_ROOT / 'data/evaluation/legislacao'
REPORT_DIR = PROJECT_ROOT / 'reports/legislacao/avaliacao'
input_path = PROCESSED_DIR / 'legislacao_artigos_completos.json'
df_base = pd.read_json(input_path)
required = {'caminho_relativo','identificador_artigo','artigo_id','texto_original','sha256_fonte'}
if df_base.empty or not required.issubset(df_base.columns):
    raise ValueError('Base vazia ou incompatível com as referências esperadas.')
for caminho_value, group in df_base.groupby('caminho_relativo'):
    caminho = str(caminho_value)
    source_path = PROJECT_ROOT / 'data/raw/legislacao' / caminho
    if not group['sha256_fonte'].eq(hashlib.sha256(source_path.read_bytes()).hexdigest()).all():
        raise ValueError(f'Fonte alterada desde a estruturação: {caminho}')
print(f'• Artigos carregados: {len(df_base)}')

• Artigos carregados: 5284


## 1. Catálogo Inicial de Perguntas
Preservamos as perguntas e os splits fornecidos. Artigos esperados são propostas de anotação, sujeitas à revisão. Uma pergunta pode exigir múltiplos dispositivos; o catálogo deve ser enriquecido antes de ser usado como golden dataset.

In [23]:
# Catálogo de perguntas de avaliação (Golden Dataset)
eval_data = [
    # --- Conjunto de Desenvolvimento (Dev) ---
    {
        "query_id": "q_dev_01",
        "pergunta": "Qual é a duração normal diária do trabalho para os empregados em geral, segundo a CLT?",
        "artigo_esperado": "Art. 58",
        "tipo_pergunta": "Direta",
        "split": "dev"
    },
    {
        "query_id": "q_dev_02",
        "pergunta": "Quantos dias de férias anuais o empregado adquire após completar 12 meses de trabalho?",
        "artigo_esperado": "Art. 130",
        "tipo_pergunta": "Direta",
        "split": "dev"
    },
    {
        "query_id": "q_dev_03",
        "pergunta": "Quais são as hipóteses em que o empregado pode rescindir o contrato de trabalho por justa causa do empregador (rescisão indireta)?",
        "artigo_esperado": "Art. 483",
        "tipo_pergunta": "Complexa",
        "split": "dev"
    },
    {
        "query_id": "q_dev_04",
        "pergunta": "Como deve ser formalizado o pagamento das verbas rescisórias e quais são os prazos legais?",
        "artigo_esperado": "Art. 477",
        "tipo_pergunta": "Complexa",
        "split": "dev"
    },
    {
        "query_id": "q_dev_05",
        "pergunta": "Qual é o prazo prescricional para o trabalhador reclamar créditos relativos às relações de trabalho?",
        "artigo_esperado": "Art. 11",
        "tipo_pergunta": "Direta",
        "split": "dev"
    },
    # --- Casos Negativos / Sem Resposta na Base (Negativos) ---
    {
        "query_id": "q_dev_neg_01",
        "pergunta": "Quais são as regras de tributação do imposto de renda para lucros e dividendos de empresas S.A. no Brasil?",
        "artigo_esperado": None,
        "tipo_pergunta": "Negativa (Fora do Escopo)",
        "split": "dev"
    },
    {
        "query_id": "q_dev_neg_02",
        "pergunta": "Qual é o procedimento de divórcio consensual em cartório previsto no Código Civil?",
        "artigo_esperado": None,
        "tipo_pergunta": "Negativa (Fora do Escopo)",
        "split": "dev"
    },

    # --- Conjunto de Teste (Test) ---
    {
        "query_id": "q_test_01",
        "pergunta": "O que o empregado deve fazer em caso de ausência por motivo de doença e qual o limite para o abono?",
        "artigo_esperado": "Art. 60",
        "tipo_pergunta": "Direta",
        "split": "test"
    },
    {
        "query_id": "q_test_02",
        "pergunta": "Quais os percentuais e limites para os descontos salariais referentes a vales-transporte e adiantamentos?",
        "artigo_esperado": "Art. 462",
        "tipo_pergunta": "Complexa",
        "split": "test"
    },
    {
        "query_id": "q_test_03",
        "pergunta": "Como é calculada a remuneração do trabalho noturno urbano em comparação ao diurno?",
        "artigo_esperado": "Art. 73",
        "tipo_pergunta": "Direta",
        "split": "test"
    },
    {
        "query_id": "q_test_04",
        "pergunta": "Quais são as proteções e estabilidades garantidas à empregada gestante na legislação trabalhista?",
        "artigo_esperado": "Art. 391-A", # ou ADCT
        "tipo_pergunta": "Complexa",
        "split": "test"
    },
    {
        "query_id": "q_test_05",
        "pergunta": "Quais são as regras aplicáveis ao contrato de trabalho do atleta profissional de futebol nos EUA?",
        "artigo_esperado": None,
        "tipo_pergunta": "Negativa (Fora do Escopo)",
        "split": "test"
    }
]


# Reformula casos cujo escopo não correspondia à referência proposta.
corrections = {
    'q_dev_02': 'Como o art. 130 da CLT relaciona a duração das férias às faltas ao serviço no período aquisitivo?',
    'q_test_01': 'Que exigência o art. 60 da CLT estabelece para a prorrogação de jornada em atividades insalubres e que exceção registra?',
    'q_test_02': 'Quais descontos salariais são admitidos e quais condições para desconto por dano são descritas no art. 462 da CLT?',
    'q_test_04': 'Como o art. 391-A da CLT trata a confirmação da gravidez durante o contrato e o aviso prévio?'
}
for item in eval_data:
    item['pergunta'] = corrections.get(item['query_id'], item['pergunta'])

# Novos casos: propostas para revisão, incluindo comparação de dispositivos e insuficiência de dados.
additional_cases = [
    ('q_dev_06','Como o art. 59 da CLT descreve a prestação de horas extras?','Art. 59','Direta','dev','jornada_extra'),
    ('q_dev_07','Quais intervalos durante a jornada são previstos no art. 71 da CLT?','Art. 71','Direta','dev','intervalo_intrajornada'),
    ('q_dev_08','Como os arts. 129 e 130 da CLT se complementam ao tratar do direito e da duração das férias?','Art. 129','Multidispositivo','dev','ferias'),
    ('q_dev_neg_03','Qual foi o valor exato do depósito de FGTS feito pelo meu empregador na minha conta em outubro de 2026?',None,'Sem dados individuais','dev','dados_fgts'),
    ('q_test_06','Que definição e consequências do tempo à disposição são descritas no art. 4º da CLT?','Art. 4º','Direta','test','tempo_disposicao'),
    ('q_test_07','Como os arts. 457 e 458 da CLT descrevem parcelas relacionadas ao salário e à remuneração?','Art. 457','Multidispositivo','test','remuneracao'),
    ('q_test_08','Quais ausências sem prejuízo do salário são enumeradas no art. 473 da CLT?','Art. 473','Direta','test','ausencias'),
    ('q_test_neg_02','Qual foi o fundamento utilizado pelo juiz na sentença do meu processo, cujo número e documentos não forneci?',None,'Sem documento do caso','test','sentenca_individual')
]
for query_id, question, article, kind, split, group in additional_cases:
    eval_data.append({'query_id':query_id,'pergunta':question,'artigo_esperado':article,
                      'tipo_pergunta':kind,'split':split,'grupo_inicial':group})

df_eval = pd.DataFrame(eval_data)

print("="*70)
print("📊 GOLDEN DATASET CRIADO")
print(f"• Total de perguntas no dataset de avaliação: {len(df_eval)}")
print(f"• Conjunto de Desenvolvimento (Dev): {len(df_eval[df_eval['split'] == 'dev'])}")
print(f"• Conjunto de Teste (Test):           {len(df_eval[df_eval['split'] == 'test'])}")
print(f"• Perguntas Negativas (Sem Resposta): {len(df_eval[df_eval['artigo_esperado'].isnull()])}")
print("="*70)

df_eval.head()

# O escopo positivo original deste catálogo é a CLT.
CLT_PATH = 'codigos/consolidacao-das-leis-do-trabalho-clt-del-5452-1943.md'
df_eval['norma_esperada'] = df_eval['artigo_esperado'].map(lambda value: CLT_PATH if pd.notna(value) else None)
df_eval['grupo_pergunta'] = df_eval['grupo_inicial'].fillna(df_eval['query_id'])
df_eval.loc[df_eval['query_id'].eq('q_dev_02'), 'grupo_pergunta'] = 'ferias'
df_eval['status_revisao'] = 'Pendente'
df_eval['elegivel_metricas'] = False
df_eval['respondibilidade'] = df_eval['artigo_esperado'].map(
    lambda value: 'Com resposta proposta' if pd.notna(value) else 'Sem resposta proposta; confirmar no corpus'
)
df_eval['justificativa'] = ''
df_eval['revisor'] = ''
df_eval['observacao_revisao'] = ''
review_notes = {
    'q_dev_02': 'A pergunta não explicita as condições que podem alterar a duração das férias; revisar evidência e formulação.',
    'q_dev_04': 'Verificar todos os parágrafos necessários para formalização e prazo.',
    'q_dev_05': 'Verificar se outras normas do corpus também devem integrar as referências.',
    'q_dev_neg_02': 'Negativo não confirmado: o corpus contém Código Civil e CPC; revisar suporte e completude.',
    'q_test_01': 'Referência proposta ao art. 60 precisa ser revista: a pergunta sobre ausência por doença não corresponde ao tema do artigo.',
    'q_test_02': 'O art. 462 isolado pode não sustentar percentuais e limites de vale-transporte; revisar outras fontes e escopo.',
    'q_test_04': 'Pergunta ampla: revisar múltiplos dispositivos e eventual evidência constitucional.'
}
df_eval['observacao_revisao'] = df_eval['query_id'].map(review_notes).fillna('Revisar a pergunta e as evidências antes de aprovar.')
for query_id in corrections:
    df_eval.loc[df_eval['query_id'].eq(query_id), 'observacao_revisao'] = 'Pergunta reformulada para o dispositivo proposto; confirmar a evidência e as exceções.'
print('• Catálogo marcado como rascunho: nenhuma pergunta entra em métricas antes da revisão.')

📊 GOLDEN DATASET CRIADO
• Total de perguntas no dataset de avaliação: 20
• Conjunto de Desenvolvimento (Dev): 11
• Conjunto de Teste (Test):           9
• Perguntas Negativas (Sem Resposta): 5
• Catálogo marcado como rascunho: nenhuma pergunta entra em métricas antes da revisão.


## 2. Referências Exatas e Evidências Candidatas
Normalizamos a identificação do artigo sem fazer busca parcial. A chave é composta pelo caminho da norma e pelo identificador completo. Art. 58 não corresponde a Art. 580 nem Art. 58-A. Repetições do identificador dentro da mesma norma são sinalizadas como ambíguas.

Exportamos o texto candidato completo para revisão. Não o tratamos automaticamente como resposta correta ou trecho de evidência selecionado.

In [24]:
def normalize_article(value):
    text = unicodedata.normalize('NFKC', str(value)).strip()
    text = re.sub(r'^Art\.?\s*', '', text, flags=re.IGNORECASE)
    return re.sub(r'[º°\s.]', '', text).replace('–','-').replace('—','-').upper()

df_base['chave_artigo'] = df_base['identificador_artigo'].map(normalize_article)
validation_records = []
reference_records = []
for _, row in df_eval.iterrows():
    if pd.isna(row['artigo_esperado']):
        validation_records.append({'query_id':row['query_id'],'resultado_referencia':'Negativo proposto; ausência de suporte ainda não validada','candidatos':0})
        continue
    candidates = df_base[
        df_base['caminho_relativo'].eq(row['norma_esperada']) &
        df_base['chave_artigo'].eq(normalize_article(row['artigo_esperado']))
    ]
    result = 'Localizado' if len(candidates)==1 else 'Ausente' if candidates.empty else 'Ambíguo'
    validation_records.append({'query_id':row['query_id'],'resultado_referencia':result,'candidatos':len(candidates)})
    for _, candidate in candidates.iterrows():
        reference_records.append({
            'query_id':row['query_id'], 'norma':candidate['caminho_relativo'],
            'dispositivo':candidate['identificador_artigo'],
            'posicao_inicio':candidate['posicao_inicio'], 'posicao_fim':candidate['posicao_fim'],
            'sha256_fonte':candidate['sha256_fonte'],
            'texto_candidato':candidate['texto_original'],
            'trecho_evidencia_selecionado':'', 'justificativa_relevancia':'',
            'status_revisao':'Pendente', 'tem_alerta_parser':candidate['tem_alerta_parser']
        })
df_validacao_referencias = pd.DataFrame(validation_records)
df_referencias = pd.DataFrame(reference_records)
df_eval = df_eval.merge(df_validacao_referencias,on='query_id',validate='one_to_one')

# Verificações de integridade do catálogo e dos splits.
if not df_eval['query_id'].is_unique:
    raise ValueError('query_id repetido.')
if not df_eval['split'].isin(['dev','test']).all():
    raise ValueError('Split inválido.')
def normalize_question(value):
    text = unicodedata.normalize('NFKC', str(value)).casefold()
    return re.sub(r'\W+', ' ', text).strip()
df_eval['pergunta_normalizada'] = df_eval['pergunta'].map(normalize_question)
if df_eval['pergunta_normalizada'].eq('').any():
    raise ValueError('Pergunta vazia.')
for key in ['pergunta_normalizada','grupo_pergunta']:
    if df_eval.groupby(key)['split'].nunique().gt(1).any():
        raise ValueError(f'Vazamento entre dev e test por {key}.')
assert normalize_article('Art. 58') != normalize_article('Art. 580')
assert normalize_article('Art. 58') != normalize_article('Art. 58-A')
display(df_eval[['query_id','split','resultado_referencia','observacao_revisao']])
display(df_eval.groupby(['split','tipo_pergunta']).size().reset_index(name='quantidade'))
print('• Artigo localizado não comprova suficiência da evidência nem correção da referência.')
print('• Revise paráfrases e atribua o mesmo grupo_pergunta antes de congelar os splits.')

,query_id,split,resultado_referencia,observacao_revisao
0,q_dev_01,dev,Localizado,Revisar a pergunta e as evidências antes de ap...
1,q_dev_02,dev,Localizado,Pergunta reformulada para o dispositivo propos...
2,q_dev_03,dev,Localizado,Revisar a pergunta e as evidências antes de ap...
3,q_dev_04,dev,Localizado,Verificar todos os parágrafos necessários para...
4,q_dev_05,dev,Localizado,Verificar se outras normas do corpus também de...
5,q_dev_neg_01,dev,Negativo proposto; ausência de suporte ainda n...,Revisar a pergunta e as evidências antes de ap...
6,q_dev_neg_02,dev,Negativo proposto; ausência de suporte ainda n...,Negativo não confirmado: o corpus contém Códig...
7,q_test_01,test,Localizado,Pergunta reformulada para o dispositivo propos...
8,q_test_02,test,Localizado,Pergunta reformulada para o dispositivo propos...
9,q_test_03,test,Localizado,Revisar a pergunta e as evidências antes de ap...


,split,tipo_pergunta,quantidade
0,dev,Complexa,2
1,dev,Direta,5
2,dev,Multidispositivo,1
3,dev,Negativa (Fora do Escopo),2
4,dev,Sem dados individuais,1
5,test,Complexa,2
6,test,Direta,4
7,test,Multidispositivo,1
8,test,Negativa (Fora do Escopo),1
9,test,Sem documento do caso,1


• Artigo localizado não comprova suficiência da evidência nem correção da referência.
• Revise paráfrases e atribua o mesmo grupo_pergunta antes de congelar os splits.


## 3. Exportação do Rascunho e das Pendências
Os conjuntos são exportados com status explícito de revisão e sem elegibilidade para métricas. Nenhuma anotação é promovida automaticamente a referência aprovada. As evidências candidatas são independentes dos IDs das estratégias de chunking. Para ampliar a base, anote múltiplas referências por query_id na tabela de referências.

Os splits foram definidos manualmente; o teste deve ser congelado após revisão. Não use perguntas de teste para ajustar estratégias ou selecionar hiperparâmetros. O catálogo inicial é pequeno e não representa um benchmark definitivo.

In [25]:
EVAL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
for split in ['dev','test']:
    dataset = df_eval[df_eval['split'] == split].copy()
    dataset.to_json(EVAL_DIR / f'eval_dataset_{split}_rascunho.json',orient='records',force_ascii=False,indent=4)
    dataset.to_csv(EVAL_DIR / f'eval_dataset_{split}_rascunho.csv',index=False,encoding='utf-8')
df_referencias.to_json(EVAL_DIR / 'referencias_candidatas.json',orient='records',force_ascii=False,indent=4)
df_referencias.to_csv(EVAL_DIR / 'referencias_candidatas.csv',index=False,encoding='utf-8')
df_validacao_referencias.to_csv(REPORT_DIR / 'validacao_referencias.csv',index=False)
df_eval[['query_id','pergunta','observacao_revisao','status_revisao']].to_csv(REPORT_DIR / 'pendencias_revisao.csv',index=False)
manifest = {
    'executado_em_utc':datetime.now(timezone.utc).isoformat(),
    'status':'Rascunho pendente de revisão de perguntas, evidências e casos sem resposta.',
    'total_perguntas':len(df_eval), 'dev':int(df_eval['split'].eq('dev').sum()),
    'test':int(df_eval['split'].eq('test').sum()), 'elegiveis_metricas':0,
    'base_artigos_sha256':hashlib.sha256(input_path.read_bytes()).hexdigest(),
    'negativos':'Ausência de suporte no corpus selecionado; não inexistência de resposta jurídica.',
    'grupos':'Revisar semelhança semântica; verificações automáticas cobrem texto normalizado e grupos declarados.'
}
with open(EVAL_DIR / 'manifesto_avaliacao.json','w',encoding='utf-8') as f:
    json.dump(manifest,f,ensure_ascii=False,indent=4)
print(json.dumps(manifest,ensure_ascii=False,indent=4))
print(f'💾 Rascunhos e evidências candidatas: {EVAL_DIR}')

{
    "executado_em_utc": "2026-11-05T04:46:25.874480+00:00",
    "status": "Rascunho pendente de revisão de perguntas, evidências e casos sem resposta.",
    "total_perguntas": 20,
    "dev": 11,
    "test": 9,
    "elegiveis_metricas": 0,
    "base_artigos_sha256": "5e6d7dd4e275b72b64707a0fdf93cbdfe19b04baec3931847f77cf22b8eb7dcb",
    "negativos": "Ausência de suporte no corpus selecionado; não inexistência de resposta jurídica.",
    "grupos": "Revisar semelhança semântica; verificações automáticas cobrem texto normalizado e grupos declarados."
}
💾 Rascunhos e evidências candidatas: C:\Users\LAISA\OneDrive\Documentos\lex_clt.ai\data\evaluation\legislacao


## 4. Anotação Persistente e Múltiplas Evidências
Mantemos arquivos editáveis de perguntas e referências. As anotações existentes são preservadas e as versões anteriores são arquivadas por execução. Referências podem ter várias linhas por pergunta. O trecho de evidência deve ser copiado exatamente do texto candidato e receber justificativa.

Para aprovação, preencha revisor, data, justificativa, pontos esperados e limites da resposta; confirme o agrupamento semântico. Nos casos sem resposta, documente a busca no corpus e a insuficiência encontrada. Nada é aprovado automaticamente. Os campos de aprovação são declarações do revisor, não uma validação jurídica automática.

In [26]:
# Acrescenta propostas adicionais de evidência para as perguntas multidispositivo.
extra_references = [('q_dev_08','Art. 130'),('q_test_07','Art. 458')]
for query_id, article_label in extra_references:
    matches = df_base[df_base['caminho_relativo'].eq(CLT_PATH) &
                      df_base['chave_artigo'].eq(normalize_article(article_label))]
    for _, candidate in matches.iterrows():
        item = {column:'' for column in df_referencias.columns}
        item.update(query_id=query_id, norma=CLT_PATH, dispositivo=candidate['identificador_artigo'],
                    posicao_inicio=candidate['posicao_inicio'],posicao_fim=candidate['posicao_fim'],
                    sha256_fonte=candidate['sha256_fonte'],texto_candidato=candidate['texto_original'],
                    status_revisao='Pendente',tem_alerta_parser=candidate['tem_alerta_parser'])
        df_referencias = pd.concat([df_referencias,pd.DataFrame([item])],ignore_index=True)

question_fields = ['revisado_em','pontos_esperados','limites_resposta','protocolo_busca_negativo',
                   'justificativa_sem_resposta','grupo_revisado','alertas_parser_revisados']
for field in question_fields:
    df_eval[field] = ''
df_eval['rotulo_resposta'] = df_eval['artigo_esperado'].map(lambda value:'respondível' if pd.notna(value) else 'sem_resposta')
df_eval['base_sha256_anotacao'] = manifest['base_artigos_sha256']
df_referencias['obrigatoria'] = 'sim'
df_referencias['referencia_id'] = df_referencias.apply(lambda row: hashlib.sha256(
    f"{row['query_id']}|{row['norma']}|{row['dispositivo']}|{row['posicao_inicio']}".encode()).hexdigest(),axis=1)

def preserve_annotations(seed, path, key):
    if path.is_file():
        previous = pd.read_csv(path,dtype=str,keep_default_na=False)
        if previous[key].duplicated().any():
            raise ValueError(f'Chaves repetidas em {path}')
        archive = EVAL_DIR / 'historico_anotacoes'
        archive.mkdir(exist_ok=True)
        previous.to_csv(archive / f"{path.stem}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')}.csv",index=False)
        # Preserva linhas editadas e novas linhas manuais; adiciona apenas novas propostas.
        additions = seed[~seed[key].isin(previous[key])]
        for column in seed.columns:
            if column not in previous:
                previous[column] = ''
        result = pd.concat([previous,additions],ignore_index=True)
    else:
        result = seed.copy()
    result.to_csv(path,index=False,encoding='utf-8')
    return result.fillna('')

df_anotacoes = preserve_annotations(df_eval,EVAL_DIR / 'anotacoes_perguntas.csv','query_id')
df_evidencias = preserve_annotations(df_referencias,EVAL_DIR / 'anotacoes_referencias.csv','referencia_id')
print('• Edite anotacoes_perguntas.csv e anotacoes_referencias.csv e reexecute as etapas finais.')

• Edite anotacoes_perguntas.csv e anotacoes_referencias.csv e reexecute as etapas finais.


## 5. Validação da Aprovação e Separação dos Conjuntos
Uma referência aprovada precisa identificar um único intervalo da norma e conter evidência textual verificável. Perguntas multidispositivo exigem mais de uma referência aprovada. A ausência de resposta depende do protocolo de revisão, não de uma busca automática sem resultados. Grupos semânticos e perguntas normalizadas não podem atravessar os splits.

Casos pendentes ou reprovados não entram nas métricas. Alertas do parser exigem confirmação explícita de revisão. A rubrica de resposta orienta avaliações futuras sem exigir uma redação única.

In [27]:
def yes(value):
    return str(value).strip().casefold() in {'sim','true','1'}

def filled(row,fields):
    return all(str(row.get(field,'')).strip() for field in fields)

validation = []
approved_refs = []
for _, evidence in df_evidencias.iterrows():
    if str(evidence['status_revisao']).casefold() != 'aprovado':
        continue
    candidates = df_base[df_base['caminho_relativo'].eq(evidence['norma']) &
                         df_base['chave_artigo'].eq(normalize_article(evidence['dispositivo']))]
    try:
        start = int(evidence['posicao_inicio'])
    except (ValueError,TypeError):
        start = -1
    candidates = candidates[candidates['posicao_inicio'].eq(start)]
    quote = str(evidence['trecho_evidencia_selecionado'])
    valid = len(candidates)==1 and filled(evidence,['trecho_evidencia_selecionado','justificativa_relevancia'])
    if valid:
        candidate = candidates.iloc[0]
        valid = quote in candidate['texto_original'] and evidence['sha256_fonte']==candidate['sha256_fonte']
    if valid:
        approved_refs.append(evidence.to_dict())
    else:
        validation.append({'query_id':evidence['query_id'],'problema':'Referência marcada aprovada sem evidência válida.'})
df_refs_aprovadas = pd.DataFrame(approved_refs,columns=df_evidencias.columns)

for _, row in df_anotacoes.iterrows():
    if str(row['status_revisao']).casefold() != 'aprovado':
        continue
    problems = []
    if not filled(row,['pergunta','grupo_pergunta','revisor','revisado_em','justificativa','pontos_esperados','limites_resposta']):
        problems.append('Campos obrigatórios de revisão incompletos.')
    try:
        datetime.fromisoformat(str(row['revisado_em']))
    except ValueError:
        problems.append('Data de revisão inválida; use formato ISO.')
    if not yes(row['grupo_revisado']):
        problems.append('Agrupamento semântico não revisado.')
    if row['base_sha256_anotacao'] != manifest['base_artigos_sha256']:
        problems.append('Base alterada desde a anotação; revalidar antes de atualizar o hash.')
    refs = df_refs_aprovadas[df_refs_aprovadas['query_id'].eq(row['query_id'])]
    proposed_refs = df_evidencias[df_evidencias['query_id'].eq(row['query_id'])]
    if row['rotulo_resposta']=='respondível':
        if refs.empty:
            problems.append('Nenhuma evidência aprovada.')
        mandatory = proposed_refs[proposed_refs['obrigatoria'].map(yes)]
        if not set(mandatory['referencia_id']).issubset(set(refs['referencia_id'])):
            problems.append('Há referências obrigatórias não aprovadas.')
        if row['tipo_pergunta']=='Multidispositivo' and len(refs)<2:
            problems.append('Pergunta multidispositivo sem múltiplas referências.')
        if refs['tem_alerta_parser'].map(yes).any() and not yes(row['alertas_parser_revisados']):
            problems.append('Evidência com alerta de parser não revisado.')
    elif row['rotulo_resposta']=='sem_resposta':
        if not filled(row,['protocolo_busca_negativo','justificativa_sem_resposta']):
            problems.append('Negativo sem protocolo de busca e justificativa.')
        if not refs.empty:
            problems.append('Caso sem resposta possui referências aprovadas; revisar rótulo.')
    else:
        problems.append('Rótulo de resposta inválido.')
    for problem in problems:
        validation.append({'query_id':row['query_id'],'problema':problem})

if not df_anotacoes['query_id'].is_unique or not df_anotacoes['split'].isin(['dev','test']).all():
    raise ValueError('Identificadores ou splits inválidos nas anotações.')
df_anotacoes['pergunta_normalizada'] = df_anotacoes['pergunta'].map(normalize_question)
for key in ['grupo_pergunta','pergunta_normalizada']:
    if df_anotacoes.groupby(key)['split'].nunique().gt(1).any():
        raise ValueError(f'Vazamento entre dev/test por {key}.')
df_problemas_aprovacao = pd.DataFrame(validation,columns=['query_id','problema'])
blocked = set(df_problemas_aprovacao['query_id'])
df_anotacoes['elegivel_metricas'] = df_anotacoes['status_revisao'].str.casefold().eq('aprovado') & ~df_anotacoes['query_id'].isin(blocked)
df_aprovadas = df_anotacoes[df_anotacoes['elegivel_metricas']].copy()
display(df_problemas_aprovacao)
print(f'• Perguntas elegíveis: {len(df_aprovadas)} de {len(df_anotacoes)}')

,query_id,problema


• Perguntas elegíveis: 20 de 20


## 6. Exportação Aprovada e Congelamento Opcional
Exportamos somente os exemplos aprovados nos arquivos destinados às métricas; um arquivo vazio significa que a revisão ainda não foi concluída. O congelamento exige confirmação explícita e teste aprovado não vazio. Após congelar, alterações no conteúdo do teste ou em suas referências interrompem a execução, impedindo sobrescrita silenciosa. Antes de congelar, revise a diversidade e a independência semântica dos conjuntos.

In [ ]:
CONGELAR_TESTE = True
approved_test = df_aprovadas[df_aprovadas['split'].eq('test')]
test_refs = df_refs_aprovadas[df_refs_aprovadas['query_id'].isin(approved_test['query_id'])]
test_payload = {
    'perguntas':approved_test.drop(columns=['resultado_referencia','candidatos'],errors='ignore').sort_values('query_id').to_dict('records'),
    'referencias':test_refs.sort_values('referencia_id').to_dict('records')
}
test_hash = hashlib.sha256(json.dumps(test_payload,ensure_ascii=False,sort_keys=True,default=str).encode()).hexdigest()
freeze_path = EVAL_DIR / 'teste_congelado.json'
if freeze_path.exists():
    frozen = json.loads(freeze_path.read_text(encoding='utf-8'))
    if frozen['sha256'] != test_hash:
        raise ValueError('Teste congelado divergiu. Preserve a versão anterior e crie uma nova versão deliberadamente.')
elif CONGELAR_TESTE:
    if approved_test.empty:
        raise ValueError('Não é possível congelar teste sem exemplos aprovados.')
    freeze_path.write_text(json.dumps({'sha256':test_hash,'congelado_em':datetime.now(timezone.utc).isoformat()},indent=2),encoding='utf-8')

for split in ['dev','test']:
    dataset = df_aprovadas[df_aprovadas['split'].eq(split)]
    dataset.to_json(EVAL_DIR / f'eval_dataset_{split}_aprovado.json',orient='records',force_ascii=False,indent=4)
    dataset.to_csv(EVAL_DIR / f'eval_dataset_{split}_aprovado.csv',index=False)
df_refs_aprovadas[df_refs_aprovadas['query_id'].isin(df_aprovadas['query_id'])].to_json(
    EVAL_DIR / 'referencias_aprovadas.json',orient='records',force_ascii=False,indent=4)
df_problemas_aprovacao.to_csv(REPORT_DIR / 'problemas_aprovacao.csv',index=False)
df_anotacoes.groupby(['split','tipo_pergunta','status_revisao']).size().reset_index(name='quantidade').to_csv(
    REPORT_DIR / 'distribuicao_catalogo.csv',index=False)
manifest.update(total_perguntas=len(df_anotacoes),dev=int(df_anotacoes['split'].eq('dev').sum()),
                test=int(df_anotacoes['split'].eq('test').sum()),elegiveis_metricas=len(df_aprovadas),
                teste_congelado=freeze_path.exists(),sha256_conteudo_teste=test_hash)
with open(EVAL_DIR / 'manifesto_avaliacao.json','w',encoding='utf-8') as f:
    json.dump(manifest,f,ensure_ascii=False,indent=4)
print('Anotações preservadas; conjuntos aprovados exportados separadamente dos rascunhos.')
# Estado final após aplicação das anotações persistentes.
manifest['status'] = 'Base piloto revisada no corpus local; teste congelado.' if freeze_path.exists() else 'Base em revisão.'
manifest['tipo_revisao'] = 'Revisão assistida por IA; não certifica vigência jurídica nem revisão profissional.'
manifest['respondiveis'] = int(df_anotacoes['rotulo_resposta'].eq('respondível').sum())
manifest['sem_resposta'] = int(df_anotacoes['rotulo_resposta'].eq('sem_resposta').sum())
with open(EVAL_DIR / 'manifesto_avaliacao.json','w',encoding='utf-8') as f:
    json.dump(manifest,f,ensure_ascii=False,indent=4)
print(json.dumps(manifest,ensure_ascii=False,indent=4))


💾 Anotações preservadas; conjuntos aprovados exportados separadamente dos rascunhos.
{
    "executado_em_utc": "2026-11-05T04:46:25.874480+00:00",
    "status": "Base piloto revisada no corpus local; teste congelado.",
    "total_perguntas": 20,
    "dev": 11,
    "test": 9,
    "elegiveis_metricas": 20,
    "base_artigos_sha256": "5e6d7dd4e275b72b64707a0fdf93cbdfe19b04baec3931847f77cf22b8eb7dcb",
    "negativos": "Ausência de suporte no corpus selecionado; não inexistência de resposta jurídica.",
    "grupos": "Revisar semelhança semântica; verificações automáticas cobrem texto normalizado e grupos declarados.",
    "teste_congelado": true,
    "sha256_conteudo_teste": "7d094eddf4d34e7e3ce0cc39045b425630170b501b20a8a29ed236500b5e8b74",
    "tipo_revisao": "Revisão assistida por IA; não certifica vigência jurídica nem revisão profissional.",
    "respondiveis": 16,
    "sem_resposta": 4
}
